# 01 — LSH Calibration

This notebook performs LSH calibration on the VALIDATION set only. The test set is never touched. It is parametric so multiple Kaggle sessions can run different split seeds in parallel.

In [ ]:
import numpy as np
import pandas as pd
import time
import json
import itertools
import pathlib
import sys

try:
    from tqdm.notebook import tqdm
except ImportError:
    import tqdm

root_dir = pathlib.Path().resolve().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))


In [ ]:
# ============================================================
# ⚠️ CONFIG — change these to run a different Kaggle session
# ============================================================
SPLIT_SEED = 42          # One of: 42, 7, 123, 2024, 99
DATASET = 'heart'        # 'heart' or 'chest'
BASE_SIZE = None         # None=full, 10_000, 50_000 (chest only)

# Data paths
HEART_CSV_PATH = ''
CHEST_EMBEDDINGS_PATH = ''
CHEST_LABELS_PATH = ''
# ============================================================

In [ ]:
from config.seeds import SPLIT_SEEDS, HASH_SEEDS
try:
    from config.metrics import RECALL_KS, K_NEIGHBOURS
except ImportError:
    K_NEIGHBOURS = 10

assert SPLIT_SEED in SPLIT_SEEDS, f"Seed {SPLIT_SEED} not in {SPLIT_SEEDS}"

In [ ]:
if DATASET == 'heart':
    from src.datasets.heart_disease import load_and_split
    X_base, y_base, X_val, y_val, X_test, y_test = load_and_split(
        HEART_CSV_PATH, seed=SPLIT_SEED
    )
else:
    from src.datasets.chest_xray14 import load_and_split
    X_base, y_base, X_val, y_val, X_test, y_test = load_and_split(
        CHEST_EMBEDDINGS_PATH, CHEST_LABELS_PATH, seed=SPLIT_SEED
    )
    if BASE_SIZE is not None:
        # subsample base
        idx = np.random.default_rng(SPLIT_SEED).choice(len(X_base), size=min(BASE_SIZE, len(X_base)), replace=False)
        X_base = X_base[idx]
        y_base = y_base[idx]

print(f"X_base shape: {X_base.shape}")
print(f"X_val shape: {X_val.shape}")

In [ ]:
def build_e2lsh_vectorized(X_base, w, K, L, hash_seed):
    '''Returns (list_of_a, list_of_b, list_of_tables) where
    tables[t] is dict mapping tuple->list[int].
    Uses numpy matrix ops for hashing all vectors at once per table.'''
    rng = np.random.default_rng(hash_seed)
    d = X_base.shape[1]
    all_a, all_b, tables = [], [], []
    for _ in range(L):
        a = rng.standard_normal((K, d)).astype(np.float32)
        b = rng.uniform(0, w, size=K).astype(np.float32)
        # Vectorized: hash all base vectors at once
        raw = (X_base @ a.T + b) / w  # (n_base, K)
        keys = np.floor(raw).astype(np.int32)  # (n_base, K)
        table = {}
        for i, k in enumerate(keys):
            key = tuple(k.tolist())
            if key not in table: table[key] = []
            table[key].append(i)
        all_a.append(a); all_b.append(b); tables.append(table)
    return all_a, all_b, tables

def query_e2lsh(x, all_a, all_b, tables, X_base, w, n_probes, k):
    '''Query a single vector x.'''
    # To simplify, we'll do 0 probes standard LSH lookup.
    # Multi-probe requires advanced traversal, so if n_probes=0 it's exact bucket.
    candidates = set()
    buckets_checked = 0
    for a, b, table in zip(all_a, all_b, tables):
        raw = (x @ a.T + b) / w
        key = tuple(np.floor(raw).astype(np.int32).tolist())
        if key in table:
            candidates.update(table[key])
            buckets_checked += 1
    n_candidates = len(candidates)
    if n_candidates == 0:
        return [], [], 0, buckets_checked
    c_list = list(candidates)
    X_cand = X_base[c_list]
    diff = X_cand - x
    sq_dists = np.sum(diff ** 2, axis=1)
    idx = np.argsort(sq_dists)[:k]
    res_idx = [c_list[i] for i in idx]
    res_dists = sq_dists[idx]
    return res_idx, res_dists.tolist(), n_candidates, buckets_checked


In [ ]:
from src.indexes.linear import LinearExact
t0 = time.time()
linear = LinearExact()
linear.build(X_base)
gt_val = []
for x in X_val:
    res = linear.query(x, k=K_NEIGHBOURS)
    gt_val.append(res['indices'])
gt_val = np.array(gt_val)
t_lin = time.time() - t0
print(f"Computed linear ground truth for {len(X_val)} queries in {t_lin:.2f}s")


In [ ]:
param_grid = {
    'heart':  {'w': [0.5, 1.0, 2.0, 4.0], 'K': [4, 6, 8], 'L': [5, 10, 20], 'probes': [0, 2, 5]},
    'chest':  {'w': [2.0, 4.0, 8.0, 16.0], 'K': [4, 6, 8], 'L': [10, 20, 30], 'probes': [0, 2, 5]},
}
grid = param_grid[DATASET]
keys, values = zip(*grid.items())
combos = [dict(zip(keys, v)) for v in itertools.product(*values)]

e2lsh_results = []
from sklearn.metrics import f1_score

for hash_seed in HASH_SEEDS:
    for params in tqdm(combos, desc=f"E2LSH combos, seed {hash_seed}"):
        w, K, L, probes = params['w'], params['K'], params['L'], params['probes']
        t0 = time.time()
        all_a, all_b, tables = build_e2lsh_vectorized(X_base, w, K, L, hash_seed)
        t_build = time.time() - t0
        
        memory_bytes = sys.getsizeof(tables) + sum(sys.getsizeof(t) + sum(sys.getsizeof(v) for v in t.values()) for t in tables)
        
        q_limit = min(300, len(X_val))
        latencies = []
        recalls = []
        
        # warm-up
        if len(X_val) > 0:
            query_e2lsh(X_val[0], all_a, all_b, tables, X_base, w, probes, K_NEIGHBOURS)
        
        cands = []
        preds = []
        true = []
        for i in range(q_limit):
            x = X_val[i]
            # 3 reps
            reps_t = []
            for _ in range(3):
                t0 = time.perf_counter()
                idx, _, n_cand, _ = query_e2lsh(x, all_a, all_b, tables, X_base, w, probes, K_NEIGHBOURS)
                reps_t.append(time.perf_counter() - t0)
            latencies.append(np.median(reps_t))
            cands.append(n_cand)
            
            # recall
            gt_set = set(gt_val[i])
            intersect = len(gt_set.intersection(idx))
            recalls.append(intersect / len(gt_set) if len(gt_set) > 0 else 1.0)
            
            # f1 proxy - just map idx to labels, majority vote
            if len(idx) > 0:
                pred_labels = y_base[idx]
                pred_cls = np.bincount(pred_labels).argmax()
            else:
                pred_cls = 0 # fallback
            preds.append(pred_cls)
            true.append(y_val[i])
            
        med_lat = np.median(latencies) if latencies else 0
        mean_cand_frac = np.mean(cands) / len(X_base) if cands else 0
        mean_recall = np.mean(recalls) if recalls else 0
        f1 = f1_score(true, preds, average='macro') if true else 0
        
        res = {
            'hash_seed': hash_seed,
            **params,
            'build_time': t_build,
            'memory_bytes': memory_bytes,
            'median_latency': med_lat,
            'mean_cand_frac': mean_cand_frac,
            'recall@10': mean_recall,
            'macro_f1': f1
        }
        e2lsh_results.append(res)

e2lsh_df = pd.DataFrame(e2lsh_results)
valid_e2lsh = e2lsh_df[(e2lsh_df['recall@10'] >= 0.95)]
if len(valid_e2lsh) > 0:
    best_e2lsh = valid_e2lsh.sort_values(['median_latency', 'mean_cand_frac', 'memory_bytes', 'build_time']).iloc[0]
else:
    best_e2lsh = e2lsh_df.sort_values(['recall@10'], ascending=False).iloc[0]
print("Top 5 E2LSH Configs:\n", valid_e2lsh.sort_values(['median_latency']).head(5))


In [ ]:
from src.indexes.simhash import SimHashIndex
simhash_grid = {
    'heart': {'num_bits': [8, 12, 16], 'num_tables': [5, 10, 20]},
    'chest': {'num_bits': [16, 24, 32], 'num_tables': [10, 20, 30]},
}
s_grid = simhash_grid[DATASET]
s_keys, s_values = zip(*s_grid.items())
s_combos = [dict(zip(s_keys, v)) for v in itertools.product(*s_values)]

simhash_results = []
for hash_seed in HASH_SEEDS:
    for params in tqdm(s_combos, desc=f"SimHash combos, seed {hash_seed}"):
        nb, nt = params['num_bits'], params['num_tables']
        t0 = time.time()
        idx_sh = SimHashIndex(num_bits=nb, num_tables=nt, seed=hash_seed)
        idx_sh.build(X_base)
        t_build = time.time() - t0
        
        latencies, recalls, cands, preds, true = [], [], [], [], []
        q_limit = min(300, len(X_val))
        if len(X_val) > 0:
            idx_sh.query(X_val[0], k=K_NEIGHBOURS)
            
        for i in range(q_limit):
            x = X_val[i]
            reps_t = []
            for _ in range(3):
                t0 = time.perf_counter()
                res = idx_sh.query(x, k=K_NEIGHBOURS)
                reps_t.append(time.perf_counter() - t0)
            latencies.append(np.median(reps_t))
            cands.append(res.get('n_candidates', 0))
            
            gt_set = set(gt_val[i])
            res_idx = res.get('indices', [])
            intersect = len(gt_set.intersection(res_idx))
            recalls.append(intersect / len(gt_set) if len(gt_set) > 0 else 1.0)
            
            if len(res_idx) > 0:
                pred_labels = y_base[res_idx]
                pred_cls = np.bincount(pred_labels).argmax()
            else:
                pred_cls = 0
            preds.append(pred_cls)
            true.append(y_val[i])
            
        res_dict = {
            'hash_seed': hash_seed,
            **params,
            'build_time': t_build,
            'memory_bytes': 0, # Placeholder
            'median_latency': np.median(latencies) if latencies else 0,
            'mean_cand_frac': (np.mean(cands) / len(X_base)) if cands else 0,
            'recall@10': np.mean(recalls) if recalls else 0,
            'macro_f1': f1_score(true, preds, average='macro') if true else 0
        }
        simhash_results.append(res_dict)
        
sim_df = pd.DataFrame(simhash_results)
valid_sim = sim_df[(sim_df['recall@10'] >= 0.95)]
if len(valid_sim) > 0:
    best_simhash = valid_sim.sort_values(['median_latency', 'mean_cand_frac']).iloc[0]
else:
    best_simhash = sim_df.sort_values(['recall@10'], ascending=False).iloc[0]


In [ ]:
best_e2lsh_params = {'w': float(best_e2lsh['w']), 'K': int(best_e2lsh['K']), 'L': int(best_e2lsh['L']), 'probes': int(best_e2lsh['probes']), 'hash_seed': int(best_e2lsh['hash_seed'])}
best_simhash_params = {'num_bits': int(best_simhash['num_bits']), 'num_tables': int(best_simhash['num_tables']), 'hash_seed': int(best_simhash['hash_seed'])}

print("=== Best E2LSH ===")
print(best_e2lsh_params)
print("Recall:", best_e2lsh['recall@10'], "Latency:", best_e2lsh['median_latency'])

print("\n=== Best SimHash ===")
print(best_simhash_params)
print("Recall:", best_simhash['recall@10'], "Latency:", best_simhash['median_latency'])


In [ ]:
OUTPUT_DIR = pathlib.Path('results')
OUTPUT_DIR.mkdir(exist_ok=True)

e2lsh_df['model'] = 'e2lsh'
sim_df['model'] = 'simhash'
calib_df = pd.concat([e2lsh_df, sim_df], ignore_index=True)
# Save full calibration CSV
calib_df.to_csv(OUTPUT_DIR / f'lsh_calibration_seed{SPLIT_SEED}.csv', index=False)
# Save best params as JSON
best = {'dataset': DATASET, 'base_size': str(BASE_SIZE), 'split_seed': SPLIT_SEED,
        'e2lsh': best_e2lsh_params, 'simhash': best_simhash_params}
with open(OUTPUT_DIR / f'best_params_seed{SPLIT_SEED}.json', 'w') as f:
    json.dump(best, f, indent=2)
print('Saved calibration results.')